In [ ]:
import re
import unicodedata

with open("../data/processed/prideprejudice.txt", "r", encoding="utf-8") as f:
    text1 = f.read()

with open("../data/processed/cthulhu.txt", "r", encoding="utf-8") as f:
    text2 = f.read()

text1 = unicodedata.normalize("NFKC", text1)
text2 = unicodedata.normalize("NFKC", text2)

text1 = re.sub(r"\[Illustration:.*?\]\]?", "", text1, flags=re.DOTALL)
text1 = re.sub(r"_([^_]+)_", r"\1", text1)
text2 = re.sub(r"_([^_]+)_", r"\1", text2)
text1 = re.sub(r"\n\s*CHAPTER\s+[IVXLC0-9]+\.*\s*\n", "\n", text1, flags=re.IGNORECASE)
text2 = re.sub(r"\n\s*CHAPTER\s+[IVXLC0-9]+\.*\s*\n", "\n", text2, flags=re.IGNORECASE)

text1 = re.sub(r"\n+", " ", text1)
text1 = re.sub(r"\s+", " ", text1)
text1 = text1.strip()
text2 = re.sub(r"\n+", " ", text2)
text2 = re.sub(r"\s+", " ", text2)
text2 = text2.strip()



words1 = text1.split()
chunks1 = []
for i in range (0, len(words1), 160):
    chunk = words1[i:i+160]
    if len(chunk) >= 120:
        chunks1.append(" ".join(chunk))

words2 = text2.split()
chunks2 = []
for i in range (0, len(words2), 160):
    chunk = words2[i:i+160]
    if len(chunk) >= 120:
        chunks2.append(" ".join(chunk))

with open("../data/processed/prideprejudice.txt", "w", encoding="utf-8") as f:
    f.write(text1)

with open("../data/processed/cthulhu.txt", "w", encoding="utf-8") as f:
    f.write(text2)

with open("../data/processed/prideprejudicechunked.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks1))

with open("../data/processed/cthulhuchunked.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks2))


In [ ]:
import google.generativeai as genai
import time
import os
from getpass import getpass

# Configure Gemini API
api_key = os.getenv("GEMINI_API_KEY")

# If not in environment, prompt the user to enter it
if not api_key:
    print("GEMINI_API_KEY not found in environment.")
    print("Paste your API key below (it will be hidden):")
    api_key = getpass("API Key: ")

if not api_key:
    raise ValueError("API key is required to continue.")

genai.configure(api_key=api_key)

# Read topics
with open("../data/processed/cthulhutopics.txt", "r") as f:
    cthulhu_topics = [line.strip() for line in f if line.strip()]

with open("../data/processed/prideprejudicetopics.txt", "r") as f:
    pride_prejudice_topics = [line.strip() for line in f if line.strip()]

print(f"Cthulhu topics: {len(cthulhu_topics)}")
print(f"Pride and Prejudice topics: {len(pride_prejudice_topics)}")
print("API configured successfully!")


In [ ]:
# Class 2: Generate 500 paragraphs (100-200 words) - Generic style
def generate_class2_paragraphs(topics, output_file, num_paragraphs):
    """Generate paragraphs without style constraint"""
    paragraphs = []
    model = genai.GenerativeModel('gemini-2.0-flash')
    
    # Create a topics string
    topics_str = ", ".join(topics)
    
    for i in range(num_paragraphs):
        try:
            prompt = f"Write a paragraph (100-200 words) that touches upon these topics: {topics_str}. Do not output any intro text, just the paragraph."
            response = model.generate_content(prompt)
            paragraphs.append(response.text)
            print(f"Generated {len(paragraphs)}/{num_paragraphs} paragraphs")
            time.sleep(1)  # Rate limiting
        except Exception as e:
            print(f"Error generating paragraph: {e}")
            time.sleep(5)
    
    # Save to file
    os.makedirs("../data/generated", exist_ok=True)
    with open(output_file, "w") as f:
        f.write("\n\n".join(paragraphs))
    
    print(f"Saved {len(paragraphs)} paragraphs to {output_file}")

# Class 3: Generate 500 paragraphs with style mimicking
def generate_class3_paragraphs(topics, output_file, author_style, num_paragraphs):
    """Generate paragraphs mimicking a specific author's style"""
    paragraphs = []
    model = genai.GenerativeModel('gemini-2.0-flash')
    
    # Create a topics string
    topics_str = ", ".join(topics)
    
    for i in range(num_paragraphs):
        try:
            prompt = f"Write a paragraph (100-200 words) that touches upon these topics: {topics_str} in the style of {author_style}"
            response = model.generate_content(prompt)
            paragraphs.append(response.text)
            print(f"Generated {len(paragraphs)}/{num_paragraphs} paragraphs")
            time.sleep(1)  # Rate limiting
        except Exception as e:
            print(f"Error generating paragraph: {e}")
            time.sleep(5)
    
    # Save to file
    os.makedirs("../data/generated", exist_ok=True)
    with open(output_file, "w") as f:
        f.write("\n\n".join(paragraphs))
    
    print(f"Saved {len(paragraphs)} paragraphs to {output_file}")


In [4]:
# Generate Class 2 paragraphs (generic style)
print("Generating Class 2 paragraphs for Cthulhu topics...")
generate_class2_paragraphs(cthulhu_topics, "../data/generated/cthulhu_class2.txt", num_paragraphs=500)

print("\nGenerating Class 2 paragraphs for Pride and Prejudice topics...")
generate_class2_paragraphs(pride_prejudice_topics, "../data/generated/prideprejudice_class2.txt", num_paragraphs=500)

# Generate Class 3 paragraphs (author style-mimicking)
print("Generating Class 3 paragraphs for Cthulhu topics (H.P. Lovecraft style)...")
generate_class3_paragraphs(cthulhu_topics, "../data/generated/cthulhu_class3.txt", 
                          author_style="H.P. Lovecraft with cosmic horror and gothic atmosphere", 
                          num_paragraphs=500)

print("\nGenerating Class 3 paragraphs for Pride and Prejudice topics (Jane Austen style)...")
generate_class3_paragraphs(pride_prejudice_topics, "../data/generated/prideprejudice_class3.txt", 
                          author_style="Jane Austen with wit, social commentary, and romantic intrigue", 
                          num_paragraphs=500)

print("\nAll paragraphs generated successfully!")



Generating Class 2 paragraphs for Cthulhu topics...
Generated 1/500 paragraphs
Generated 2/500 paragraphs
Generated 3/500 paragraphs
Generated 4/500 paragraphs
Generated 5/500 paragraphs
Generated 6/500 paragraphs
Generated 7/500 paragraphs
Generated 8/500 paragraphs
Generated 9/500 paragraphs
Generated 10/500 paragraphs
Generated 11/500 paragraphs
Generated 12/500 paragraphs
Generated 13/500 paragraphs
Generated 14/500 paragraphs
Generated 15/500 paragraphs
Generated 16/500 paragraphs
Generated 17/500 paragraphs
Error generating paragraph: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.
Error generating paragraph: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.
Generated 18/500 paragraphs
Generated 19/500 paragraphs
Generated 20/500 paragraphs
Generated 21/500 paragraphs
Error gener

KeyboardInterrupt: 

In [ ]:
# Generate Class 3 paragraphs (author style-mimicking)
print("Generating Class 3 paragraphs for Cthulhu topics (H.P. Lovecraft style)...")
generate_class3_paragraphs(cthulhu_topics, "../data/generated/cthulhu_class3.txt", 
                          author_style="H.P. Lovecraft with cosmic horror and gothic atmosphere", 
                          num_paragraphs=500)

print("\nGenerating Class 3 paragraphs for Pride and Prejudice topics (Jane Austen style)...")
generate_class3_paragraphs(pride_prejudice_topics, "../data/generated/prideprejudice_class3.txt", 
                          author_style="Jane Austen with wit, social commentary, and romantic intrigue", 
                          num_paragraphs=500)

print("\nAll paragraphs generated successfully!")
